In [1]:
from ax import Client
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
gp_model = Client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-2_BO4ACST/ExperimentalSeries-2_TOoUPRCS/Method20250625Dim3_CurSys_qLNEI_UCSvYM/AxExperiment.json")
gp_model_df = gp_model.summarize()
gp_model.get_next_trials(max_trials=1)

{71: {'x1': 0.45, 'x2': 0.4, 'x3': 0.5879253794774315}}

In [3]:
Prior_QR_Naive_x1_arr = np.array(gp_model_df["x1"][0:8])
Prior_QR_Naive_x2_arr = np.array(gp_model_df["x2"][0:8])
Prior_QR_Naive_x3_arr = np.array(gp_model_df["x3"][0:8])
Prior_QR_Naive_y1_arr = np.array(gp_model_df["ucs"][0:8])
Prior_QR_Naive_y2_arr = np.array(gp_model_df["ym"][0:8])
Prior_QR_Naive_X_mat = np.array([Prior_QR_Naive_x1_arr,Prior_QR_Naive_x2_arr,Prior_QR_Naive_x3_arr])
Prior_QR_Naive_Y_mat = np.array([Prior_QR_Naive_y1_arr,Prior_QR_Naive_y2_arr])

In [4]:
def SurrogateModelOfReality(x1, x2, x3):
    y_pred = gp_model.predict([{"x1":x1,"x2":x2,"x3":x3}])[0]["ucs"][0]
    return np.float64(y_pred)

In [5]:
def SOBO():
    # Takes all that it needs
    # Returns a sequential attempts results:
    # A = the sample number at which the baseline was exceeded
    # B = the best sample obtained during the run

    client = Client()
    parameters = [RangeParameterConfig(name="x1", parameter_type="float", bounds=(0.45, 1)),
                    RangeParameterConfig(name="x2", parameter_type="float", bounds=(0.4, 1)),
                    RangeParameterConfig(name="x3", parameter_type="float", bounds=(0.5, 0.95))]
    client.configure_experiment(parameters=parameters)

    def construct_generation_strategy(generator_spec: GeneratorSpec, node_name: str,) -> GenerationStrategy:
        botorch_node = GenerationNode(node_name=node_name,model_specs=[generator_spec],)
        return GenerationStrategy(name=f"{node_name}",nodes=[botorch_node])

    construct_generation_strategy(generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),node_name="Modular BoTorch")
    surrogate_spec = SurrogateSpec(model_configs=[ModelConfig(botorch_model_class=SingleTaskGP,covar_module_class=MaternKernel,covar_module_options={"nu": 2.5},),],eval_criterion=MSE,allow_batched_models=False,)

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            "acquisition_options": {},
        },
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(generator_spec=generator_spec,node_name="BoTorch w/ Model Selection")
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1"
    objective = f"{metric_name}"
    client.configure_optimization(objective=objective)

    sampler = Sampler_class()
    X = sampler.three.QuasirandomSampler3D_func(NoSD_flt=8,Parameters_lis=parameters).T

    for array in X:
        my_parameters = {"x1": array[0], "x2": array[1], "x3": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    Initial_df = client.summarize()
    Baseline_ys = np.max(Initial_df["t1"])

    for _ in range(16):
        trials = client.get_next_trials(max_trials=1)
        for trial_index, parameters in trials.items():
            x1 = parameters["x1"]
            x2 = parameters["x2"]
            x3 = parameters["x3"]
            result = SurrogateModelOfReality(x1,x2,x3)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    Final_df = client.summarize()
    Best_ys = np.max(Final_df["t1"])
    i_breach = np.max(Final_df.index.tolist())+1

    for i,CurrentYS in enumerate(Final_df["t1"][8::]):
        if CurrentYS > Baseline_ys:
            i_breach = i
            break
        
    return Best_ys,i_breach

ys_max_flt,i_breach_flt = SOBO()

In [6]:
ys_max_arr = np.empty(0)
i_breach_arr = np.empty(0)
runs = 100
for i in range(runs):
    ys_max_flt,i_breach_flt = SOBO()
    ys_max_arr = np.append(ys_max_arr,ys_max_flt)
    i_breach_arr = np.append(i_breach_arr,i_breach_flt)
    print(f"{i+1}/{runs}")
print(round(np.average(ys_max_arr),2))
print(round(np.average(i_breach_arr),2))

0/100
1/100
2/100
3/100
4/100
5/100
6/100
7/100
8/100
9/100
10/100
11/100
12/100
13/100
14/100
15/100
16/100
17/100
18/100


/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


19/100
20/100
21/100
22/100
23/100
24/100
25/100
26/100
27/100
28/100
29/100
30/100
31/100
32/100
33/100
34/100
35/100
36/100
37/100
38/100
39/100
40/100
41/100
42/100
43/100
44/100
45/100
46/100
47/100
48/100
49/100
50/100
51/100
52/100
53/100
54/100
55/100
56/100
57/100
58/100
59/100
60/100
61/100
62/100
63/100
64/100
65/100
66/100
67/100
68/100
69/100
70/100
71/100
72/100
73/100
74/100
75/100
76/100
77/100
78/100
79/100
80/100


/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


81/100
82/100
83/100
84/100
85/100
86/100
87/100
88/100
89/100
90/100
91/100
92/100
93/100
94/100


/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


95/100
96/100
97/100
98/100
99/100
138.65
0.7


In [8]:
def MIPT():
    parameters = [RangeParameterConfig(name="x1", parameter_type="float", bounds=(0.45, 1)),
                    RangeParameterConfig(name="x2", parameter_type="float", bounds=(0.4, 1)),
                    RangeParameterConfig(name="x3", parameter_type="float", bounds=(0.5, 0.95))]
    sampler = Sampler_class()
    X = sampler.three.QuasirandomSampler3D_func(NoSD_flt=8,Parameters_lis=parameters).T
    ȳ = np.empty(0)
    for array in X:
        y = SurrogateModelOfReality(x1=array[0],x2=array[1],x3=array[2])
        ȳ = np.append(ȳ,y)
    Baseline_ys = np.max(ȳ)
    for _ in range(16):
        DimensionNames_lis = []
        for i in parameters:
            DimensionNames_lis.append(i.name)
        b_lis = []
        for i in parameters:
            b_lis.append(i.bounds)
        d_flt = len(b_lis)
        n_flt = len(X.T[0])
        c_flt = 100*n_flt
        alpha_flt = 0.5
        y_lis = []
        for i in b_lis:
            y_lis.append(np.random.uniform(i[0],i[1],c_flt))
        y_mat = np.array(y_lis)
        d_min_flt = (2*alpha_flt)/n_flt
        y_results = []
        for y_row in y_mat.T:
            gobbles_lis = []
            degooks_lis = []
            for x_row in X:
                gobbles_lis.append(np.linalg.norm(x_row-y_row,ord=np.inf))
                degooks_lis.append(np.linalg.norm(x_row-y_row,ord=2))
            gobble_flt = np.min(gobbles_lis)
            degook_flt = degooks_lis[np.argmin(gobbles_lis)]
            if gobble_flt < d_min_flt:
                y_results.append(0)
            else:
                y_results.append(degook_flt)
        y_coord_lis = []
        for dimension in y_mat:
            y_coord_lis.append([dimension[np.argmax(y_results)]])
        y_coord_mat = np.array(y_coord_lis)
        Putative_x1x2x3_arr = y_coord_mat.T[0]
        y = SurrogateModelOfReality(x1=Putative_x1x2x3_arr[0],x2=Putative_x1x2x3_arr[1],x3=Putative_x1x2x3_arr[2])
        ȳ = np.append(ȳ,y)
        Best_ys = np.max(ȳ)
        i_breach = len(ȳ)
        for i,CurrentYS in enumerate(ȳ[8::]):
            if CurrentYS > Baseline_ys:
                i_breach = i
                break
            
        return Best_ys,i_breach
    
MIPT()

(np.float64(134.482375363721), 9)

In [9]:
ys_max_arr = np.empty(0)
i_breach_arr = np.empty(0)
runs=100
for i in range(runs):
    ys_max_flt,i_breach_flt = MIPT()
    ys_max_arr = np.append(ys_max_arr,ys_max_flt)
    i_breach_arr = np.append(i_breach_arr,i_breach_flt)
    print(f"{i+1}/{runs}")
print(round(np.average(ys_max_arr),2))
print(round(np.average(i_breach_arr),2))

1/100
2/100
3/100
4/100
5/100
6/100
7/100
8/100
9/100
10/100
11/100
12/100
13/100
14/100
15/100
16/100
17/100
18/100
19/100
20/100
21/100
22/100
23/100
24/100
25/100
26/100
27/100
28/100
29/100
30/100
31/100
32/100
33/100
34/100
35/100
36/100
37/100
38/100
39/100
40/100
41/100
42/100
43/100
44/100
45/100
46/100
47/100
48/100
49/100
50/100
51/100
52/100
53/100
54/100
55/100
56/100
57/100
58/100
59/100
60/100
61/100
62/100
63/100
64/100
65/100
66/100
67/100
68/100
69/100
70/100
71/100
72/100
73/100
74/100
75/100
76/100
77/100
78/100
79/100
80/100
81/100
82/100
83/100
84/100
85/100
86/100
87/100
88/100
89/100
90/100
91/100
92/100
93/100
94/100
95/100
96/100
97/100
98/100
99/100
100/100
133.41
8.55


In [ ]:

X = np.array()

In [ ]:
def MOBO():

In [ ]:
np.linspace
def SurrogateModelOfReality(x1, x2, x3):
    y_pred = gp_model.predict([{"x1":x1,"x2":x2,"x3":x3}])[0]["ucs"][0]
    return np.float64(y_pred)